# Sampling timing parameters with Discovery and Enterprise

This notebook is the everyday path: bind a `TimingSpec` to a pulsar, sample with Discovery (NumPyro) and Enterprise (PTMCMC), switch backends, and plot the chain.

`decentered_model` is the everyday Discovery sampler model; notebook 3 explains the alternative. Charts and geometry wait until notebooks 2 and 4.

A `TimingPulsar` is required: an object that implements the necessary attributes (a protocol) Today that is MetaPulsar; the `nltiming` calls do not change if Discovery or Enterprise grow a native host.


In [ ]:
import os
import sys
os.environ.setdefault("JAX_ENABLE_X64", "1")

import jax
import numpy as np
import matplotlib.pyplot as plt
import corner
import discovery as ds
from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, load_run
from nltiming.sampling import numpyro
from nltiming.sampling import ptmcmc

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="WARNING")

numpyro.ensure_x64()


## A TimingPulsar

AEI-DR2 combined J1721-2457 (isolated, 150 TOAs). Run this notebook from `examples/notebooks/`.


In [ ]:
DATA = Path("..") / "data" / "J1721-2457"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1721-2457.par",
        "tim": DATA / "J1721-2457.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
print(pulsar.name, len(pulsar.toas))


## Discovery

Default inference samples the nonlinear axes (here proper motion) and analytically marginalizes the rest. White noise is the same EFAC-only model as Enterprise, using Discovery's default `Uniform(0.1, 10)`. Short pedagogical chain — scale `num_warmup` / `num_samples` for science.


In [ ]:
spec = TimingSpec(engines="jug", name="timing")
timing = spec.for_pulsar(pulsar)
print("sampled:", timing.sampled)
print("marginalized:", timing.marginalized)

efac = f"{pulsar.name}_efac"
EFAC_PRIOR = (0.1, 10.0)  # Discovery default: "(.*_)?efac" in discovery.prior.priordict_standard
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, add_equad=False),
    *timing.discovery_signals(),
])
model = numpyro.decentered_model(
    likelihood, timing, priors={efac: EFAC_PRIOR},
)

from numpyro.infer import init_to_value
init = {
    **numpyro.decentered_init_values(timing, model.transport),
    efac: 1.0,
}
mcmc = numpyro.nuts(
    model, timing,
    num_warmup=200, num_samples=1000, num_chains=1,
    init_strategy=init_to_value(values=init),
)
mcmc.run(jax.random.PRNGKey(0))


## Physical posterior

`numpyro.posterior()` decodes the recorded timing offsets into chain-preserving ArviZ data with short parameter names and display units.


In [ ]:
post = numpyro.posterior(mcmc, timing)
corner.corner(post, var_names=list(timing.sampled))
plt.show()

## Enterprise

Same default plan, libstempo backend. Enterprise parameters are prior-normal `z` (`..._timing_PMRA`), not physical units. Raw `PTSampler` — no `enterprise_extensions`.


In [ ]:
import tempfile
from enterprise.signals import parameter, signal_base, white_signals
from PTMCMCSampler.PTMCMCSampler import PTSampler

spec_ent = spec.with_engines({"tempo2": "libstempo"})
timing_ent = spec_ent.for_pulsar(pulsar)
white = white_signals.MeasurementNoise(efac=parameter.Uniform(*EFAC_PRIOR))
pta = signal_base.PTA([(white + spec_ent.enterprise_signal())(pulsar)])
print(pta.param_names)

outdir = Path(tempfile.mkdtemp(prefix="nlt_ent_"))
timing_ent.write(
    outdir, likelihood="enterprise", sampler="ptmcmc",
    chain_layout=ptmcmc.chain_layout(timing_ent, pta.param_names),
)

x0 = np.hstack([np.asarray(p.sample(), dtype=float).reshape(-1) for p in pta.params])
sampler = PTSampler(
    len(pta.param_names), pta.get_lnlikelihood, pta.get_lnprior,
    np.diag(np.full(len(pta.param_names), 0.1**2)), outDir=str(outdir),
)
sampler.sample(x0, Niter=5000)



In [ ]:
run = load_run(outdir)
post = run.posterior(burn=0.25)
corner.corner(np.column_stack([post[k] for k in timing_ent.sampled]), labels=list(timing_ent.sampled))
plt.show()

In [ ]:
names = list(timing_ent.sampled)
ent = np.column_stack([post[k] for k in names])
disc = np.column_stack([
    np.asarray(numpyro.posterior(mcmc, timing).posterior[k]).reshape(-1)
    for k in names
])
hist_kw = {"density": True, "histtype": "step"}
fig = corner.corner(
    ent, labels=names, color="C0",
    weights=np.full(len(ent), 1.0 / len(ent)),
    hist_kwargs={**hist_kw, "color": "C0"},
)
corner.corner(
    disc, fig=fig, color="C1",
    plot_datapoints=False, plot_density=False,
    fill_contours=False, no_fill_contours=True,
    weights=np.full(len(disc), 1.0 / len(disc)),
    hist_kwargs={**hist_kw, "color": "C1"},
)
fig.legend(
    handles=[
        plt.Line2D([0], [0], color="C0", label="Enterprise"),
        plt.Line2D([0], [0], color="C1", label="Discovery"),
    ],
    loc="upper right",
)
plt.show()

In [ ]:
disc_efac = np.asarray(mcmc.get_samples()[efac]).reshape(-1)
chain = np.loadtxt(outdir / "chain_1.txt")
ent_efac = chain[int(0.25 * len(chain)) :, list(pta.param_names).index(efac)]
plt.hist(ent_efac, bins=40, density=True, histtype="step", color="C0", label="Enterprise")
plt.hist(disc_efac, bins=40, density=True, histtype="step", color="C1", label="Discovery")
plt.xlabel(efac)
plt.ylabel("density")
plt.legend()
